# Data analyisis

First we load... 

In [75]:
import pandas as pd
import os

repo_root = os.getcwd()
file_path = os.path.join(repo_root, '..', "Data Collection", "MASTER_ALL_DATA.csv")

raw_df = pd.read_csv(file_path)

# # Add a parameter that tells read_csv to not parse strings as NaN
# df_different_parsing = pd.read_csv(file_path, keep_default_na=False)

raw_df.head(3)


transition = '\n𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 \n'

/var/folders/sz/8v4khwpx0b76jkngnc80xll40000gn/T/ipykernel_56095/3150949563.py:7: DtypeWarning: Columns (0: title, 1: created_utc, 2: permalink) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(file_path)


# 2. Tidy up data format
Cause of NaN Problem: we have vertical stack of data, we need to merge the 3 data sources to have a horizontal (full) dataset to work with

--> re-separate 3 sources, then merge using app_id as the merge id

In [76]:
raw_df = pd.read_csv(file_path, dtype={'app_id': int, 'title': str, 'created_utc': str, 'permalink': str})

#Steam Monthly Player Data
steam_df = raw_df[raw_df['month'].notna()].copy()
steam_df = steam_df[['app_id', 'game_name', 'genre', 'month', 'peak_players', 'avg_players']]

# Reddit engagement data 
reddit_engagement = raw_df[raw_df['engagement'].notna()].copy()
reddit_engagement = reddit_engagement[['app_id', 'engagement', 'valence', 'n_posts']]

# Ensure one row per game app_id
reddit_engagement = reddit_engagement.drop_duplicates(subset=['app_id'])

# (Inner Join) merge using app_id, we only keep data where there is a merge partner
df_analysis = pd.merge(steam_df, reddit_engagement, on='app_id', how='inner')


print(df_analysis.head(3))
df_analysis.info()

    app_id            game_name  \
0  1063730  New World: Aeternum   
1  1063730  New World: Aeternum   
2  1063730  New World: Aeternum   

                                           genre         month peak_players  \
0  Action, Adventure, Massively Multiplayer, RPG  Last 30 Days          987   
1  Action, Adventure, Massively Multiplayer, RPG    April 2026          975   
2  Action, Adventure, Massively Multiplayer, RPG    March 2026        1,093   

  avg_players  engagement  valence  n_posts  
0       550.0         0.0      NaN      0.0  
1       579.3         0.0      NaN      0.0  
2       581.7         0.0      NaN      0.0  
<class 'pandas.DataFrame'>
RangeIndex: 443820 entries, 0 to 443819
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   app_id        443820 non-null  int64  
 1   game_name     443820 non-null  str    
 2   genre         443820 non-null  str    
 3   month         443820 non-null  str

## Set data types
e.g.


 \#   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   app_id        443820 non-null  int64            <- str
 1   game_name     443820 non-null  str              <- category                  
 2   genre         443820 non-null  str              <- category    
 3   month         443820 non-null  str              <- ?    
 4   peak_players  443820 non-null  str              <- float    
 5   avg_players   443820 non-null  str              <- float             


In [77]:
#our key shouldn't be used for mathematical operations:
df_analysis['app_id'] = df_analysis['app_id'].astype(str)

# Convert string columns to categorical data
categorical_features = ['game_name', 'genre']
df_analysis[categorical_features] = df_analysis[categorical_features].astype("category")

# Let's see how many unique values we have for each of these features
print(df_analysis[categorical_features].nunique())
df_analysis.head()

df_analysis['peak_players'] = df_analysis['peak_players'].astype(str).str.replace(',', '').astype(float)
df_analysis['avg_players'] = df_analysis['avg_players'].astype(str).str.replace(',', '').astype(float)

df_analysis.info()

game_name    1917
genre         359
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 443820 entries, 0 to 443819
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   app_id        443820 non-null  str     
 1   game_name     443820 non-null  category
 2   genre         443820 non-null  category
 3   month         443820 non-null  str     
 4   peak_players  443820 non-null  float64 
 5   avg_players   443820 non-null  float64 
 6   engagement    443820 non-null  float64 
 7   valence       207441 non-null  float64 
 8   n_posts       443820 non-null  float64 
dtypes: category(2), float64(5), str(2)
memory usage: 25.4 MB


#### Drop 'Last 30 Days' from df_analysis['month']
to keep overlaps with months from happening.

In [78]:
df_analysis['month']
df_analysis = df_analysis[df_analysis['month'] != 'Last 30 Days']

Overview 2.O

In [79]:
nb_games = df_analysis['game_name'].nunique()
genres = df_analysis['genre'].nunique()
print(f"""For analysis the cleaned dataset contains:
      
        -{nb_games} games,
        -{genres} genres,
        -{df_analysis.shape[0]} valid rows
        
        """)

#rows per game /months with peak_player data available
print(f"The top 5 games with the most data on monthly players are:\n\n{df_analysis['game_name'].value_counts().head(5)}")

For analysis the cleaned dataset contains:

        -1917 games,
        -359 genres,
        -439253 valid rows

        
The top 5 games with the most data on monthly players are:

game_name
Heroes & Generals             1224
TrackMania Nations Forever    1162
theHunter Classic             1144
Transformice                  1088
Clicker Heroes                1056
Name: count, dtype: int64


# 3. Re-Gain overview
- see what data we're dealing with at this point
- find missing values (so we can decide how to handle them)
- check completeness
- check distribution of values to see which analyses are viable


In [80]:
df = df_analysis.copy() #easier to type and if we mess up we dont affect our clean initial one

print(df.describe()) #only shows the numerical variables
print(transition)

df.info()
print(transition)


# What would that dropping NaNs do to our data?
df.dropna().info()
print(transition)




       peak_players   avg_players    engagement        valence        n_posts
count  4.392530e+05  4.392530e+05  4.392530e+05  205386.000000  439253.000000
mean   5.508656e+03  2.652918e+03  1.013499e+04       0.757790       3.837112
std    3.688176e+04  1.671862e+04  5.459929e+04       0.150853       8.573090
min    0.000000e+00  0.000000e+00  0.000000e+00       0.230000       0.000000
25%    2.100000e+02  8.130000e+01  0.000000e+00       0.681429       0.000000
50%    8.020000e+02  3.459000e+02  0.000000e+00       0.787778       0.000000
75%    2.580000e+03  1.234400e+03  8.600000e+01       0.860000       4.000000
max    3.236027e+06  1.584887e+06  1.622955e+06       1.000000     158.000000

𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 

<class 'pandas.DataFrame'>
Index: 439253 entries, 1 to 443819
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   app_id        439253 non-null  str     
 1   game_name     439253 

### Dropping all rows containing NaN shows us:
- 207441 rows are kept from the previous 443820

We see that valence is the culprit. ( 7   valence       207441 non-null  float64)

When we don't have reddit data, the value is kept empty. We cannot use any values 0-1, because that'd give it a rating on our scale: 

`valence` Reflects how positively the game is received (1.0 = unanimous upvote, 0.5 = controversial).


Therefore we will not drop the NaN, as these contain relevant information! Games can have players even when no engagement happens on r/gaming and we want to include this data for correlation between engagement and player count!




In [84]:
# Check how many posts exist when valence is NaN
missing_valence = df_analysis[df_analysis['valence'].isna()]
print("When valence is missing, what is the post count distribution?")
print(missing_valence['n_posts'].value_counts())

print(df_analysis[df_analysis['valence'].notna()]['valence'].describe())

When valence is missing, what is the post count distribution?
n_posts
0.0    233867
Name: count, dtype: int64
count    205386.000000
mean          0.757790
std           0.150853
min           0.230000
25%           0.681429
50%           0.787778
75%           0.860000
max           1.000000
Name: valence, dtype: float64


# Aggregating our data

-> one line = one game

We keep the first value from genre, engagement, valence and n_posts as will stay consistent (from our previous merge).

- `genre`: first value 
- `player peak`:  maximum value 
- `avg_players`: mean across all months averages
- `engagement `:  first value
- `valence `:  first value 
- `n_posts `:  first value

In [ ]:

grouped_df = df_analysis.groupby(['app_id', 'game_name']).agg({
    'genre': 'first',             
    'peak_players': 'max',        # maximum peak from all peaks (h1)
    'avg_players': 'mean',        # average across all months (yes, the average of the average player counts :p)
    'engagement': 'first',        # 
    'valence': 'first',           
    'n_posts': 'first'            
}).reset_index()

print(f"Grouped dataset contains {grouped_df.shape[0]} unique games.")

#sorted by peak players
print(grouped_df[['game_name', 'peak_players', 'engagement', 'valence']].sort_values('peak_players', ascending= False).head(3))
print(transition)

#sorted by engagement
print(grouped_df[['game_name', 'peak_players', 'engagement', 'valence']].sort_values('engagement', ascending= False).head(3))
print(transition)

#sorted by valence
print(grouped_df[['game_name', 'peak_players', 'engagement', 'valence']].sort_values('valence', ascending= False).head(5))

Grouped dataset contains 1921 unique games.
                game_name  peak_players  engagement   valence
1434  PUBG: BATTLEGROUNDS     3236027.0     81156.0  0.833333
616              Lost Ark     1324761.0       131.0  0.748500
69    New World: Aeternum      913027.0         0.0       NaN

𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 

                     game_name  peak_players  engagement   valence
1770  Control Ultimate Edition        7130.0   1622955.0  0.835000
1096                      DOOM       31623.0    715244.0  0.910952
112             Cyberpunk 2077      830387.0    674966.0  0.889024

𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 

          game_name  peak_players  engagement  valence
581         OpenTTD        3196.0         3.0      1.0
1514     Stoneshard       10706.0         1.0      1.0
1634  Overcooked! 2       17176.0         3.0      1.0


In [83]:
# What would that dropping NaNs do to our data?
df.dropna().info()

df.describe().style

df.info()

# # Define the features we want to change and store this list for later
# categorical_features = []
# # Let's see how many unique values we have for each of these features
# print(df[categorical_features].nunique())

# # Convert string columns to categorical data
# df[categorical_features] = df[categorical_features].astype("category")


# # Make a deep copy of our dataframe to store the original dataframe for later comparison
# df_deepfake = df.copy()

<class 'pandas.DataFrame'>
Index: 205386 entries, 121 to 443528
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   app_id        205386 non-null  str     
 1   game_name     205386 non-null  category
 2   genre         205386 non-null  category
 3   month         205386 non-null  str     
 4   peak_players  205386 non-null  float64 
 5   avg_players   205386 non-null  float64 
 6   engagement    205386 non-null  float64 
 7   valence       205386 non-null  float64 
 8   n_posts       205386 non-null  float64 
dtypes: category(2), float64(5), str(2)
memory usage: 13.3 MB
<class 'pandas.DataFrame'>
Index: 439253 entries, 1 to 443819
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   app_id        439253 non-null  str     
 1   game_name     439253 non-null  category
 2   genre         439253 non-null  category
 3   month         439253 no